### Instructions

This notebook collects and plots results for DRO-RoBAS experiments. It assumes each method has been run on 0, 10% and 20% of contamination levels.

### Imports

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### Useful functions

In [ ]:
def process(experiment_dir, dgp, contamination, trim=False):
    results_df = pd.read_csv(experiment_dir/ "results.csv", index_col=["uuid", "replication"])
    results_df = results_df.loc[(results_df["contamination"] == contamination) & (results_df["dgp"] == dgp)]
    if trim:
        results_df = results_df[results_df['epsilon'] < 0.2]  # where to trim the epsilons, adjust accordingly if needed
    gb = results_df.groupby(by=["algorithm", "epsilon", "inference"])
    agg_df = gb.agg({
    "mean_cost": ["mean", "var"],
    "var_cost": ["mean"],
    })
    return agg_df

In [ ]:
def plot_df(df, dgp: str, algorithm: str, label: str, axis, shp: str,  marker=False):
    # get a dataframe for each posterior
    df = df.loc[algorithm, :]
    # # mean of variances + variance of means
    bootstrap_var = df["var_cost"]["mean"] + df["mean_cost"]["var"]
    # plot mean-variance trade-off
    axis.plot(bootstrap_var, df["mean_cost"]["mean"], marker=shp, linestyle='solid', label=label)
    # set title and label epsilon
    axis.set_title(f"{dgp}")
    axis.set_xlabel("out-of-sample variance")
    if dgp == (r'$\eta =$' + '0.0'):
        axis.set_ylabel("out-of-sample mean")
        axis.legend()
    if marker:
        for i, epsilon in enumerate(df.index.get_level_values("epsilon")):
            if i % 4 == 0:
                axis.text(bootstrap_var[epsilon], df["mean_cost"]["mean"][epsilon], epsilon, ha='left', va='bottom')
        # if (i+2) % 4 == 0:
        #     axis.text(empirical_bootstrap_var[epsilon], empirical_df["mean_cost"]["mean"][epsilon], epsilon, ha='right', va='bottom')

### Set experiment directories for each method (or group of methods) for one DGP

In [ ]:
experiment_path_robas = Path("/dcs/pg23/u1604520/misdro/results_kdro/slurm_batches/bimodal_robas")  # Where RoBAS results are stored
experiment_path_kl = Path("/dcs/pg23/u1604520/misdro/results_kdro/slurm_batches/bimodal_bas_univ") # Where DRO-BAS results are stored
experiment_path_bdro = Path("/dcs/pg23/u1604520/misdro/results_kdro/slurm_batches/bimodal_bdro_univ") # where BDRO results are stored
experiment_path_emp = Path("/dcs/pg23/u1604520/misdro/results_kdro/slurm_batches/bimodal_robas") # Where empirical mmd results are stored
experiment_path_bdro_npl = Path("/dcs/pg23/u1604520/misdro/results_kdro/slurm_batches/bimodal_bdro_npl_univ") # where bdro npl reuslts are stored

In [ ]:
results_df = pd.read_csv(experiment_path_bdro_npl/ "results.csv", index_col=["uuid", "replication"])
results_df = results_df.loc[(results_df["contamination"] == 0.0) & (results_df["dgp"] == "bimodal_univariate_gaussian")]

In [ ]:
results_df

### Plot!

In [ ]:
fig, axis = plt.subplots(nrows=1, ncols=3, sharex=False, sharey=False, figsize=(14,4))
contam_levels = [0.0, 0.1, 0.2]  # all contamination levels

# Marker shapes
shp_bas = 'o'
shp_bdro = 's'
shp_bdro_npl = 'x'
shp_emp = '+'
shp_robas = 'v'

for i, contam in enumerate(contam_levels):
    agg_df_kl = process(experiment_path_kl, 'bimodal_univariate_gaussian', contam, trim=True)  
    agg_df_emp = process(experiment_path_emp, 'bimodal_univariate_gaussian', contam,trim=True)
    agg_df_robas = process(experiment_path_robas, 'bimodal_univariate_gaussian', contam,trim=True)
    agg_df_bdro = process(experiment_path_bdro, 'bimodal_univariate_gaussian', contam,trim=True)
    agg_df_bdro_npl = process(experiment_path_bdro_npl, 'bimodal_univariate_gaussian', contam,trim=True)
    plot_df(agg_df_kl, r'$\eta =$' + f'{contam}', 'kl_dro_bas', 'DRO-BAS', axis[i], shp_bas)
    plot_df(agg_df_bdro, r'$\eta =$' + f'{contam}', 'kl_bdro', 'BDRO', axis[i], shp_bdro, marker=True)
    plot_df(agg_df_bdro_npl, r'$\eta =$' + f'{contam}', 'kl_bdro', 'BDRO-NPL', axis[i], shp_bdro_npl)
    plot_df(agg_df_emp, r'$\eta =$' + f'{contam}', 'empirical_mmd', 'MMD-empirical', axis[i], shp_emp)
    plot_df(agg_df_robas,r'$\eta =$' + f'{contam}', 'dro_bas_mmd', 'RoBAS', axis[i], shp_robas, marker=True)